# Experimentos reproducibles con estrategias de ruleta

## Objetivo

Ejecutar múltiples partidas en condiciones comparables y analizar
los resultados de cada estrategia.

## Preguntas del experimento

- ¿Qué proporción de partidas termina con ganancias?
- ¿Cuál es el beneficio o la pérdida media por partida?
- ¿Cuánto varían los resultados entre partidas?
- ¿Con qué frecuencia el saldo resulta insuficiente para realizar
  la siguiente apuesta?

## Organización de los datos

Mantendremos dos tablas:

- Tiradas: una fila por tirada, identificando a qué partida pertenece.
- Partidas: una fila por partida, con su configuración, resultados
  y motivo de finalización.

Una partida puede terminar sin realizar ninguna tirada.
En ese caso aparecerá en la tabla de partidas, pero no en la de tiradas.

## Reproducibilidad

Registraremos la configuración y controlaremos la aleatoriedad
para poder repetir un experimento bajo las mismas condiciones.

## Alcance del primer experimento

Compararemos inicialmente martingala y la variante de paroli
implementada en el script.

Configuración común:

- Saldo inicial: 100 €.
- Apuesta inicial y de reinicio: 1 €.
- Apuesta siempre a par.
- Máximo de 100 tiradas por partida.
- Sin límites voluntarios de ganancias o pérdidas.

Reglas de las estrategias:

- Martingala: duplica la apuesta después de perder y vuelve
  a la apuesta inicial después de ganar.
- Paroli: duplica la apuesta después de ganar y vuelve
  a la apuesta inicial después de perder.

El script también conserva su límite numérico de seguridad.

Labouchere inversa se incorporará en una fase posterior:
actualmente utiliza la secuencia [1, 2, 3, 4] y empieza apostando 5 €.

In [4]:
initial_balance = 100
base_bet = 1
max_spins = 100
side = "par"
strategies = ["martingala", "paroli"]

In [5]:
import pandas as pd

results = pd.read_csv(
    "session_001.csv",
    dtype={"session_id": "string"}
)

results.head()

,session_id,random_seed,strategy,spin,side,number,bet,balance_before,won,profit,balance_after
0,session_001,42,martingala,1,par,12,1,100,1,1,101
1,session_001,42,martingala,2,par,6,1,101,1,1,102
2,session_001,42,martingala,3,par,29,1,102,0,-1,101
3,session_001,42,martingala,4,par,5,2,101,0,-2,99
4,session_001,42,martingala,5,par,11,4,99,0,-4,95


In [6]:
results[["session_id", "random_seed"]].drop_duplicates()

,session_id,random_seed
0,session_001,42


## Resumir una partida

El detalle contiene una fila por tirada.
El resumen contendrá una fila por partida.

Crearemos una función para obtener el identificador, la semilla,
la estrategia, el número de tiradas y el beneficio neto.

Esta primera versión requiere al menos una tirada.
Más adelante incorporaremos los metadatos del simulador
para representar también partidas sin tiradas.

In [11]:
def summarize_session(spins):
    if spins.empty:
        raise ValueError("The session must contain at least one spin.")

    initial_balance = int(spins["balance_before"].iloc[0])
    final_balance = int(spins["balance_after"].iloc[-1])
    net_profit = int(spins["profit"].sum())
    return_percentage = net_profit / initial_balance * 100

    return {
        "session_id": spins["session_id"].iloc[0],
        "random_seed": spins["random_seed"].iloc[0],
        "strategy": spins["strategy"].iloc[0],
        "total_spins": len(spins),
        "initial_balance": initial_balance,
        "final_balance": final_balance,
        "net_profit": net_profit,
        "return_percentage": return_percentage,
    }

In [13]:
session_summary = summarize_session(results)

session_summaries = pd.DataFrame([session_summary])

session_summaries

,session_id,random_seed,strategy,total_spins,initial_balance,final_balance,net_profit,return_percentage
0,session_001,42,martingala,91,100,20,-80,-80.0


## Incorporar una segunda partida

Ejecutamos otra partida con la misma estrategia, saldo inicial,
apuesta base y máximo de tiradas.

Cambiamos la semilla y el identificador para obtener otra ejecución
y distinguir sus datos.

Aplicamos la misma función de resumen a ambas partidas.

In [14]:
second_results = pd.read_csv(
    "session_002.csv",
    dtype={"session_id": "string"},
)

second_summary = summarize_session(second_results)

session_summaries = pd.DataFrame([
    session_summary,
    second_summary,
])

session_summaries

,session_id,random_seed,strategy,total_spins,initial_balance,final_balance,net_profit,return_percentage
0,session_001,42,martingala,91,100,20,-80,-80.0
1,session_002,43,martingala,14,100,42,-58,-58.0


In [15]:
second_results[
    ["spin", "number", "bet", "won", "balance_after"]
].tail(8)

,spin,number,bet,won,balance_after
6,7,14,2,1,104
7,8,32,1,1,105
8,9,1,1,0,104
9,10,19,2,0,102
10,11,27,4,0,98
11,12,33,8,0,90
12,13,35,16,0,74
13,14,25,32,0,42


## Observación de la segunda partida

La partida terminó tras 14 tiradas, con un saldo de 42 €.

Las seis últimas tiradas fueron derrotas, con apuestas de
1, 2, 4, 8, 16 y 32 €, que suman una pérdida de 63 €.

La siguiente apuesta habría sido de 64 €, superior al saldo
disponible. Por eso no pudo continuar.

Este caso muestra que una partida puede terminar por saldo
insuficiente sin que el saldo llegue a cero.

## Ejecutar un primer lote de partidas

Ejecutaremos 10 partidas de martingala con la misma configuración
y una semilla diferente para cada partida.

Python se encargará de:

1. Ejecutar el simulador de Bash.
2. Cargar las tiradas del CSV generado.
3. Resumir cada partida.
4. Reunir los resúmenes en una tabla.

Cada ejecución del lote tendrá su propia carpeta para conservar
los resultados anteriores.

In [16]:
import subprocess
from pathlib import Path
from tempfile import mkdtemp

script_path = Path("ruleta.sh").resolve()

if not script_path.is_file():
    raise FileNotFoundError(f"Script not found: {script_path}")

print("Script:", script_path)

Script: /home/vanest/workSpace/Machine learning-IA/bash-scripting/ruleta.sh


In [17]:
number_of_sessions = 10
first_seed = 100

batch_directory = Path(
    mkdtemp(prefix="roulette_batch_", dir=Path.cwd())
)

batch_summaries = []

print("Results directory:", batch_directory)

Results directory: /home/vanest/workSpace/Machine learning-IA/bash-scripting/roulette_batch_6j3aubml


In [20]:
for session_number in range(1, number_of_sessions + 1):
    session_id = f"{batch_directory.name}_{session_number:03d}"
    random_seed = first_seed + session_number - 1
    output_path = batch_directory / f"{session_id}.csv"

    command = [
        "bash",
        str(script_path),
        "-m", str(initial_balance),
        "-t", "martingala",
        "-b", str(base_bet),
        "-s", side,
        "-l",
        "-n", str(max_spins),
        "-d", "0",
        "-r", str(random_seed),
        "-i", session_id,
        "-o", str(output_path),
    ]

    execution = subprocess.run(
        command,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
    )

    if execution.returncode != 0:
        raise RuntimeError(
            f"Session failed: {session_id}\n"
            f"{execution.stderr}\n{execution.stdout}"
        )

    spins = pd.read_csv(
        output_path,
        dtype={"session_id": "string"},
    )

    batch_summaries.append(summarize_session(spins))

    print(
        f"Completed {session_number}/{number_of_sessions}: "
        f"{len(spins)} spins"
    )

Completed 1/10: 41 spins
Completed 2/10: 12 spins
Completed 3/10: 15 spins
Completed 4/10: 98 spins
Completed 5/10: 30 spins
Completed 6/10: 86 spins
Completed 7/10: 38 spins
Completed 8/10: 37 spins
Completed 9/10: 100 spins
Completed 10/10: 100 spins


In [21]:
batch_results = pd.DataFrame(batch_summaries)

batch_results

,session_id,random_seed,strategy,total_spins,initial_balance,final_balance,net_profit,return_percentage
0,roulette_batch_6j3aubml_001,100,martingala,41,100,49,-51,-51.0
1,roulette_batch_6j3aubml_002,101,martingala,12,100,41,-59,-59.0
2,roulette_batch_6j3aubml_003,102,martingala,15,100,40,-60,-60.0
3,roulette_batch_6j3aubml_004,103,martingala,98,100,24,-76,-76.0
4,roulette_batch_6j3aubml_005,104,martingala,30,100,50,-50,-50.0
5,roulette_batch_6j3aubml_006,105,martingala,86,100,11,-89,-89.0
6,roulette_batch_6j3aubml_007,106,martingala,38,100,53,-47,-47.0
7,roulette_batch_6j3aubml_008,107,martingala,37,100,51,-49,-49.0
8,roulette_batch_6j3aubml_009,108,martingala,100,100,150,50,50.0
9,roulette_batch_6j3aubml_010,109,martingala,100,100,137,37,37.0


## Variación de los resultados

Además de la media, calculamos:

- Mínimo: el peor resultado económico del lote.
- Máximo: el mejor resultado económico del lote.
- Mediana: el valor central al ordenar los resultados.
- Rango: diferencia entre el máximo y el mínimo.

Cuando hay un número par de partidas, la mediana se calcula
como la media de los dos valores centrales.

In [22]:
profits = batch_results["net_profit"]

minimum_profit = profits.min()
maximum_profit = profits.max()
median_profit = profits.median()
profit_range = maximum_profit - minimum_profit

print(f"Minimum net profit: {minimum_profit:.2f} EUR")
print(f"Maximum net profit: {maximum_profit:.2f} EUR")
print(f"Median net profit: {median_profit:.2f} EUR")
print(f"Net profit range: {profit_range:.2f} EUR")

Minimum net profit: -89.00 EUR
Maximum net profit: 50.00 EUR
Median net profit: -50.50 EUR
Net profit range: 139.00 EUR


## Comparar martingala y paroli

Ejecutamos ambas estrategias con las mismas condiciones:

- Saldo inicial: 100 €.
- Apuesta inicial: 1 €.
- Apuesta a par.
- Máximo de 100 tiradas.
- Sin límites voluntarios de ganancias o pérdidas.
- Semillas del 100 al 109.

Con el mismo entorno y generador, cada semilla produce la misma
secuencia de números para ambas estrategias. Las apuestas y los
saldos pueden ser distintos, y cada estrategia puede terminar antes.

La variante de paroli utilizada duplica después de ganar
y vuelve a la apuesta inicial después de perder.

In [23]:
martingale_results = batch_results.copy()

In [24]:
paroli_directory = Path(
    mkdtemp(prefix="paroli_batch_", dir=Path.cwd())
)

paroli_summaries = []

for session_number in range(1, number_of_sessions + 1):
    session_id = f"{paroli_directory.name}_{session_number:03d}"
    random_seed = first_seed + session_number - 1
    output_path = paroli_directory / f"{session_id}.csv"

    command = [
        "bash",
        str(script_path),
        "-m", str(initial_balance),
        "-t", "paroli",
        "-b", str(base_bet),
        "-s", side,
        "-l",
        "-n", str(max_spins),
        "-d", "0",
        "-r", str(random_seed),
        "-i", session_id,
        "-o", str(output_path),
    ]

    execution = subprocess.run(
        command,
        stdin=subprocess.DEVNULL,
        capture_output=True,
        text=True,
    )

    if execution.returncode != 0:
        raise RuntimeError(
            f"Session failed: {session_id}\n"
            f"{execution.stderr}\n{execution.stdout}"
        )

    spins = pd.read_csv(
        output_path,
        dtype={"session_id": "string"},
    )

    paroli_summaries.append(summarize_session(spins))

    print(
        f"Completed {session_number}/{number_of_sessions}: "
        f"{len(spins)} spins"
    )

paroli_results = pd.DataFrame(paroli_summaries)

Completed 1/10: 100 spins
Completed 2/10: 100 spins
Completed 3/10: 100 spins
Completed 4/10: 100 spins
Completed 5/10: 100 spins
Completed 6/10: 100 spins
Completed 7/10: 100 spins
Completed 8/10: 100 spins
Completed 9/10: 100 spins
Completed 10/10: 100 spins


In [25]:
profit_comparison = martingale_results[
    ["random_seed", "net_profit"]
].merge(
    paroli_results[["random_seed", "net_profit"]],
    on="random_seed",
    suffixes=("_martingale", "_paroli"),
    validate="one_to_one",
)

profit_comparison

,random_seed,net_profit_martingale,net_profit_paroli
0,100,-51,-58
1,101,-59,-53
2,102,-60,-48
3,103,-76,-44
4,104,-50,-53
5,105,-89,-55
6,106,-47,-58
7,107,-49,-57
8,108,50,-49
9,109,37,-56


## Comparación por semilla

Calculamos la diferencia entre el beneficio de paroli y el de martingala.

- Diferencia positiva: paroli obtuvo un mejor resultado.
- Diferencia negativa: martingala obtuvo un mejor resultado.
- Diferencia cero: ambas obtuvieron el mismo resultado.

Obtener un mejor resultado puede significar perder menos dinero;
no implica necesariamente terminar con ganancias.

In [26]:
profit_comparison["profit_difference"] = (
    profit_comparison["net_profit_paroli"]
    - profit_comparison["net_profit_martingale"]
)

profit_comparison

,random_seed,net_profit_martingale,net_profit_paroli,profit_difference
0,100,-51,-58,-7
1,101,-59,-53,6
2,102,-60,-48,12
3,103,-76,-44,32
4,104,-50,-53,-3
5,105,-89,-55,34
6,106,-47,-58,-11
7,107,-49,-57,-8
8,108,50,-49,-99
9,109,37,-56,-93


In [27]:
differences = profit_comparison["profit_difference"]

print("Paroli performed better:", (differences > 0).sum())
print("Martingale performed better:", (differences < 0).sum())
print("Equal results:", (differences == 0).sum())

Paroli performed better: 4
Martingale performed better: 6
Equal results: 0
